# Milestone 1 — arXiv Scraper v2 (hybrid S3 + export)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DuyNhat1705/Intro2DS-Group3-Source/blob/main/src/ExportS3/scraper_v2.ipynb)

Notebook điều phối **end-to-end** của scraper v2: entry discovery → full-text source mọi version (bulk S3 cho bài chỉ có v1, export `/src/` cho phần còn lại) → reference từ Semantic Scholar, đồng thời đo hiệu năng và xuất số liệu cho Jupyter Book.

Notebook chỉ **gọi** các module, không sửa chúng: `get_full_source_v2`, `s3_source` (cùng thư mục `src/ExportS3/`) và `get_metadata`, `get_full_source`, `get_extracting_reference`, `utils` (trong `src/`).

## §0. Giới thiệu

**Vì sao v2.** Discovery cho thấy 79,5% trong 25.000 bài chỉ có v1. Nếu tải mọi version qua `export.arxiv.org/src` (1 request mỗi version, cách nhau ≥ 3 giây) thì cần 31.530 request, khoảng 26 giờ. v2 lấy v1 của những bài đó từ **bucket S3 bulk của arXiv** (`s3://arxiv/src/`, requester pays). Bucket này không giới hạn nhịp request, nên export chỉ còn khoảng 11.700 request (~10 giờ) và hai nguồn chạy **song song**.

**Kiến trúc chạy** — một instance (Colab CPU), một tiến trình, bốn thread:

```
            ┌── Thread "arxiv" (1 kết nối, ≥ ARXIV_DELAY giây giữa lúc BẮT ĐẦU hai request) ─────────┐
 ID list ──►│ Phase A: metadata theo batch cho mọi bài ─► lập kế hoạch ─► Phase B: export /src/      │
            │                                                 │            (bài nhiều version, bài    │
            │                                                 │             S3 không có) ◄──┐        │
            └─────────────────────────────────────────────────┼─────────────────────────────┼────────┘
            ┌── Thread "s3" ──────────────────────────────────▼─────────────────────────────┼────────┐
            │ manifest ─► đọc stream từng chunk tar ─► .tex/.bib của các bài cần ─► bài thiếu ┘        │
            └──────────────────────────────────────────────────────────────────────────────────────────┘
            ┌── Thread "s2" ── Phase C: references theo batch ─────────────────────────────────────────┐
            ┌── Thread "monitor" ── RSS (psutil) mỗi MONITOR_EVERY giây, dung lượng data/ mỗi DISK_EVERY┐
```

- **Kế hoạch** (sau Phase A, khi đã biết số version): bài chỉ có v1 và nằm trong một chunk S3 thì giao cho làn S3; còn lại (bài nhiều version, bài chưa lên S3, chunk vượt `S3_MAX_GB`) đi export. `S3_LATEST_FOR_MULTI = True` thì vN của bài nhiều version cũng lấy từ S3 (chỉ bật sau khi đã kiểm chứng, xem §2).
- **Làn S3** đọc chunk theo stream, không ghi tar ra đĩa. Làn dừng đọc ngay khi đã gặp đủ các bài cần trong chunk; nếu stream đứt (hoặc phiên kết thúc) thì lần sau đọc tiếp từ ranh giới member gần nhất (HTTP Range), không trả tiền hai lần cho cùng một byte. Bài không có trong chunk, archive hỏng, hoặc S3 không dùng được (thiếu key, không có quyền) thì **trả về cho làn export**. Vì vậy scraper vẫn chạy hết kể cả khi không có S3.
- **Làn export**: dùng chung một nhịp cho API metadata và `/src/`, tính từ lúc request bắt đầu. Version chỉ có PDF được nhận ra từ `Content-Type` nên không tải body. `.tex` / `.bib` được giải nén thẳng từ RAM, archive không ghi ra đĩa. Khi gặp 429 / 5xx kéo dài, làn **nghỉ `COOLDOWN_SEC` rồi thử lại**; gặp 403 thì dừng làn.
- **Tiến độ luôn đọc từ file state trên đĩa**: `source_state.json` (giữ schema v1, nên dữ liệu v1 đã tải được dùng tiếp) và `.state/s3_chunks.json` (chunk nào đã đọc tới đâu). Bị dừng (hết giờ Colab, 403, Ctrl+C) thì chỉ cần chạy lại notebook.

**Chạy trên Google Colab** (testbed của đề)

1. Mở notebook bằng nút *Open in Colab* ở đầu trang; *Runtime → Change runtime type* → **CPU**.
2. *Secrets* (biểu tượng chìa khoá), bật *Notebook access* cho từng key:
   - `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`: IAM user có quyền đọc S3 (`AmazonS3ReadOnlyAccess`). Bucket là requester pays: **tài khoản AWS của nhóm trả phí egress** (~0,09 USD/GB, xem ước tính ở dòng `plan` khi chạy). Không có key thì làn S3 dừng ngay và mọi version đi qua export (~26 giờ).
   - `S2_API_KEY` (khuyến nghị): Semantic Scholar. Không có key vẫn chạy được nhưng dễ bị 429.
3. *Runtime → Run all*. Lần đầu của mỗi runtime, §1 clone repo và cài thư viện theo `src/requirements.txt` và `src/ExportS3/requirements.txt`; nếu notebook báo cần restart thì *Runtime → Restart session* rồi *Run all* lại.
4. §3 mount Google Drive: `data/` nằm ở `MyDrive/Intro2DS-Group3/data` để còn lại qua nhiều phiên.
5. §9 cào tới khi xong, khi đạt `MAX_RUN_HOURS` (11,5 giờ) hoặc khi dịch vụ từ chối; sau đó Phần IV tự chạy. Giữ tab mở.
6. Phiên sau: mở lại notebook và *Run all*. Lặp lại tới khi §4 báo không còn việc.

Chạy thử trước: đặt `SAMPLE_N = 20` ở §2 (20 bài đầu mỗi range).

**Chạy local (Jupyter)**: `pip install -r src/requirements.txt -r src/ExportS3/requirements.txt`, đặt `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` (biến môi trường hoặc `~/.aws/credentials`), mở notebook từ bên trong repo; `data/` nằm ở `<repo>/data`.

**Cấu trúc `data/`**

```
data/
├── <yymm-nnnnn>/            # mỗi bài một thư mục (vd 2608-04007)
│   ├── metadata.json        # get_metadata
│   ├── references.json      # get_extracting_reference (mọi reference)
│   └── v1/, v2/, ...        # chỉ .tex/.bib (rỗng nếu version chỉ có PDF)
├── source_state.json        # trạng thái từng version (via: export | s3:<chunk>), kích thước
├── .state/                  # refs_done.jsonl, nb_status.jsonl, s3_chunks.json, s3_manifest.xml
├── run_logs/                # runs.jsonl, timings.jsonl, events.jsonl, monitor_*.csv, disk_*.csv, scraper.log
├── tables/                  # papers.parquet, references.parquet, references_all.parquet
└── report_exports/          # số liệu nhỏ cho Jupyter Book
```

# Phần I — Setup

## §1. Môi trường

Tìm thư mục repo (trên Colab: clone repo), thêm `src/` vào `sys.path`, kiểm tra thư viện theo `src/requirements.txt` và ghi lại thông tin testbed.

In [ ]:
# §1 — Tìm repo (Colab: clone), thêm src/ và src/ExportS3/ vào sys.path
import os
import sys
import json
import math
import re
import time
import shutil
import platform
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DuyNhat1705/Intro2DS-Group3-Source.git"
BRANCH = "scraper"  # nhánh chứa scraper v2; đổi thành "main" sau khi merge
COLAB_REPO_DIR = Path("/content/Intro2DS-Group3-Source")


def find_repo_root(start: Path):
    """Đi ngược lên từ `start` tới thư mục chứa src/utils.py."""
    for p in (start, *start.parents):
        if (p / "src" / "utils.py").is_file():
            return p
    return None


if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(COLAB_REPO_DIR)],
                       check=True)
    REPO_ROOT = COLAB_REPO_DIR
else:
    REPO_ROOT = find_repo_root(Path.cwd().resolve())
    if REPO_ROOT is None:
        raise RuntimeError("Không tìm thấy repo (src/utils.py). Hãy mở notebook từ bên trong repo.")

SRC_DIR = REPO_ROOT / "src"
V2_DIR = SRC_DIR / "ExportS3"  # get_full_source_v2, s3_source
# Các module import utils theo nhiều kiểu (from src.utils / from utils) nên thêm cả gốc repo lẫn src/
for p in (str(V2_DIR), str(SRC_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

print("Colab:", IN_COLAB)
print("Repo :", REPO_ROOT)

In [ ]:
# §1 — Kiểm tra thư viện theo src/requirements.txt + src/ExportS3/requirements.txt (Colab: tự cài)
from importlib import metadata as importlib_metadata

REQUIREMENTS_FILES = [SRC_DIR / "requirements.txt", V2_DIR / "requirements.txt"]


def read_requirements(path: Path) -> dict:
    """{tên: phiên bản} từ các dòng `tên==phiên_bản`."""
    reqs = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.split("#", 1)[0].strip()
        if "==" in line:
            name, version = line.split("==", 1)
            reqs[name.strip()] = version.strip()
    return reqs


def installed_version(name: str):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


REQS = {n: v for f in REQUIREMENTS_FILES for n, v in read_requirements(f).items()}
mismatches = {n: (installed_version(n), v) for n, v in REQS.items() if installed_version(n) != v}

if not mismatches:
    print("Thư viện đúng phiên bản trong requirements.txt.")
elif IN_COLAB:
    already_loaded = [n for n in mismatches if n.replace("-", "_") in sys.modules]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *(a for f in REQUIREMENTS_FILES for a in ("-r", str(f)))],
                   check=True)
    if already_loaded:
        raise RuntimeError(
            f"Đã cài lại {already_loaded} (đang được import sẵn trong kernel). "
            "Vào Runtime > Restart session rồi chạy lại notebook từ đầu."
        )
    print("Đã cài:", {n: v for n, (_, v) in mismatches.items()})
else:
    print("CẢNH BÁO — khác requirements.txt {tên: (đang có, cần)}:", mismatches)
    print(f"Cài bằng: {sys.executable} -m pip install " + " ".join(f"-r {f}" for f in REQUIREMENTS_FILES))

In [ ]:
# §1 — Thông tin testbed (ghi vào runs.jsonl ở mỗi phiên)
import psutil


def cpu_model() -> str:
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def machine_info() -> dict:
    vm = psutil.virtual_memory()
    disk = shutil.disk_usage(REPO_ROOT)
    return {
        "platform": platform.platform(),
        "python": platform.python_version(),
        "colab": IN_COLAB,
        "cpu_model": cpu_model(),
        "cpu_count": os.cpu_count(),
        "ram_total_bytes": vm.total,
        "disk_total_bytes": disk.total,
        "disk_free_bytes": disk.free,
        "packages": {n: installed_version(n) for n in REQS},
    }


print(json.dumps(machine_info(), indent=2, ensure_ascii=False))

## §2. Cấu hình

**Mọi tham số nằm trong ô dưới.** Chạy thử: đặt `SAMPLE_N` (số bài đầu tiên của mỗi range) và/hoặc thu hẹp `RANGES_TO_RUN`. Chạy thật: `SAMPLE_N = None`, cả 5 range.

Key **không** ghi vào notebook: đặt biến môi trường, hoặc trên Colab thêm vào *Secrets* với tên `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `S2_API_KEY`.

`S3_LATEST_FOR_MULTI`: một chunk S3 chứa **một** bản của mỗi bài, là bản mới nhất ở thời điểm chunk được đóng gói. Nếu chunk được đóng gói sau ngày của vN thì bản đó có thể là vN, giúp bớt thêm ~5.100 request export (~4 giờ). Chưa kiểm chứng nên mặc định tắt; chạy `scraper_v2_prep/scripts/07_verify_s3_latest.py` trước, khớp hết mới bật.

In [ ]:
# §2 — Cấu hình (ô duy nhất chứa tham số)
RANGES = {                      # nhãn (thành viên phụ trách) -> (YYMM, start, end), inclusive
    "tdat": ("2609", 27834, 32833),
    "mdat": ("2609", 17834, 22833),
    "ngoc": ("2608", 14007, 19006),
    "nhat": ("2608", 4007, 9006),
    "y":    ("2511", 6709, 11708),
}
RANGES_TO_RUN = list(RANGES)    # mặc định cả 5 range (25.000 bài); chỉ thu hẹp khi chạy thử
SAMPLE_N = None                 # vd 20 = 20 bài đầu mỗi range; None = toàn bộ

RUN_METADATA = True             # Phase A
RUN_SOURCE = True               # Phase B: làn export + làn S3
RUN_REFS = True                 # Phase C
PARALLEL = True                 # True: làn S2 chạy song song với hai làn source; False: chạy sau (để so sánh)

USE_S3 = True                   # False: mọi version qua export như v1
S3_MAX_GB = 150.0               # trần tổng dung lượng đọc từ S3 qua mọi phiên (requester pays, ~0,09 USD/GB);
                                # chunk vượt trần -> các bài của nó đi export
S3_LATEST_FOR_MULTI = False     # True: vN của bài nhiều version lấy từ S3 (chỉ bật sau khi kiểm chứng, xem trên)
S3_MANIFEST_MAX_AGE_DAYS = 7.0  # tải lại manifest S3 khi bản lưu trong data/.state cũ hơn

ARXIV_DELAY = 3.0               # giây giữa lúc BẮT ĐẦU hai request arXiv liên tiếp (đề: >= 3s)
KEEP_ARCHIVE = False            # True: giữ thêm file nén cạnh .tex/.bib
METADATA_BATCH_SIZE = 100       # số bài mỗi request metadata arXiv (tối đa 100)
S2_BATCH_SIZE = 100             # số bài mỗi request batch Semantic Scholar
RETRY_NOT_FOUND = False         # True: thử lại các bài / references 'not_found' của phiên trước
MAX_CONSECUTIVE_ERRORS = 5      # số lỗi liên tiếp trước khi dừng một làn
COOLDOWN_SEC = 900.0            # arXiv trả 429/5xx kéo dài: nghỉ bấy nhiêu giây rồi thử lại cùng request
MAX_COOLDOWNS = 3               # số lần nghỉ tối đa cho một request trước khi dừng làn arXiv
MAX_RUN_HOURS = 11.5            # tự dừng gọn trước khi Colab ngắt phiên (~12h); None = không giới hạn
INTERRUPT_GRACE = 60.0          # giây chờ các làn xong việc hiện tại khi bấm Interrupt

MONITOR_EVERY = 1.0             # giây giữa hai mẫu RSS
DISK_EVERY = 300.0              # giây giữa hai lần đo dung lượng data/ (quét cả thư mục, chậm trên Drive)
PROGRESS_EVERY = 600.0          # giây giữa hai dòng tiến độ in ra notebook
SOURCE_STATE_FLUSH_EVERY = 120.0 # giây; gom các lần ghi source_state.json (0 = ghi sau mỗi version)

PERSIST_TO_DRIVE = True         # Colab: data/ nằm trên Google Drive để chạy tiếp qua nhiều phiên
DRIVE_DATA_DIR = "/content/drive/MyDrive/Intro2DS-Group3/data"
MIN_FREE_DISK_GB = 5.0          # cảnh báo khi chỗ trống dưới ngưỡng

In [ ]:
# §2 — Kiểm tra cấu hình + nạp key (S2_API_KEY phải có trước khi import get_extracting_reference)
unknown = [r for r in RANGES_TO_RUN if r not in RANGES]
if unknown:
    raise ValueError(f"Nhãn range không tồn tại: {unknown}")
if ARXIV_DELAY < 3.0:
    raise ValueError("Đề yêu cầu chờ ít nhất 3 giây giữa hai request arXiv.")
if SAMPLE_N is not None and SAMPLE_N < 1:
    raise ValueError("SAMPLE_N phải >= 1 hoặc None.")
if not 1 <= METADATA_BATCH_SIZE <= 100:
    raise ValueError("METADATA_BATCH_SIZE phải trong khoảng 1..100.")
if S3_MAX_GB is not None and S3_MAX_GB < 0:
    raise ValueError("S3_MAX_GB phải >= 0 hoặc None.")

total = 0
print("Sẽ chạy:")
for label in RANGES_TO_RUN:
    yymm, start, end = RANGES[label]
    n = end - start + 1 if SAMPLE_N is None else min(SAMPLE_N, end - start + 1)
    total += n
    print(f"  {label:5s} {yymm}.{start:05d} -> {yymm}.{end:05d}   {n} bài")
print(f"Tổng: {total} bài")


def load_secret(name: str):
    """Đặt os.environ[name] từ biến môi trường hoặc Colab Secrets; trả về nguồn (không in giá trị)."""
    if os.environ.get(name):
        return "biến môi trường"
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
            return "Colab Secrets"
    return None


S2_KEY_SOURCE = load_secret("S2_API_KEY")
print("S2_API_KEY:", S2_KEY_SOURCE or "không có (dùng pool chung, dễ bị 429)")
if USE_S3 and RUN_SOURCE:
    AWS_KEY_SOURCE = {n: load_secret(n) for n in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY")}
    print("AWS key:", AWS_KEY_SOURCE)
    if not all(AWS_KEY_SOURCE.values()):
        print("CẢNH BÁO: thiếu AWS key trong biến môi trường / Secrets. Nếu cũng không có ~/.aws/credentials "
              "thì làn S3 dừng ngay và mọi version đi qua export.")

## §3. Lưu trữ và import module

`utils.DATA_DIR` cố định là `<repo>/data`. Trên Colab, notebook biến `<repo>/data` thành **symlink sang Google Drive** (không sửa `utils.py`), để dữ liệu và state còn lại sau khi phiên Colab kết thúc.

In [ ]:
# §3 — data/ (Colab: symlink sang Drive) và các file log của notebook
DATA_DIR = REPO_ROOT / "data"

if IN_COLAB and PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = Path(DRIVE_DATA_DIR)
    target.mkdir(parents=True, exist_ok=True)
    if DATA_DIR.is_symlink():
        if DATA_DIR.resolve() != target.resolve():
            raise RuntimeError(f"{DATA_DIR} đang trỏ tới {DATA_DIR.resolve()}, không phải {target}.")
    else:
        if DATA_DIR.exists():
            if any(DATA_DIR.iterdir()):
                raise RuntimeError(f"{DATA_DIR} đã có dữ liệu local; hãy chuyển sang {target} trước.")
            DATA_DIR.rmdir()
        DATA_DIR.symlink_to(target, target_is_directory=True)
DATA_DIR.mkdir(exist_ok=True)

STATE_DIR = DATA_DIR / ".state"            # dùng chung với get_extracting_reference (refs_done.jsonl)
LOG_DIR = DATA_DIR / "run_logs"
TABLE_DIR = DATA_DIR / "tables"
EXPORT_DIR = DATA_DIR / "report_exports"
for d in (STATE_DIR, LOG_DIR, TABLE_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

RUNS_FILE = LOG_DIR / "runs.jsonl"          # một dòng start + một dòng end cho mỗi phiên
TIMINGS_FILE = LOG_DIR / "timings.jsonl"    # thời gian từng (bài, stage)
EVENTS_FILE = LOG_DIR / "events.jsonl"      # sự kiện bất thường (dừng làn, batch lỗi, ...)
SCRAPER_LOG = LOG_DIR / "scraper.log"       # log chi tiết của các module
NB_STATUS_FILE = STATE_DIR / "nb_status.jsonl"  # kết quả metadata/source do notebook ghi (dòng sau ghi đè dòng trước)
NON_CORPUS_DIRS = {".state", "run_logs", "tables", "report_exports"}

free_gb = shutil.disk_usage(DATA_DIR).free / 1e9
print(f"data/ = {DATA_DIR} -> {DATA_DIR.resolve()}")
print(f"Còn trống: {free_gb:.1f} GB")
if free_gb < MIN_FREE_DISK_GB:
    print(f"CẢNH BÁO: dưới {MIN_FREE_DISK_GB} GB.")

In [ ]:
# §3 — Import module của nhóm, cấu hình logging một lần
import logging

import arxiv
import requests

import utils
import get_metadata as gm
import get_full_source as gfs
import get_full_source_v2 as gfs2
import s3_source as s3s
import get_extracting_reference as ger

if USE_S3 and RUN_SOURCE:
    import boto3  # noqa: F401  (thiếu: pip install -r src/ExportS3/requirements.txt)

if not hasattr(gm, "get_entry_discovery_batch"):
    raise RuntimeError(
        f"{SRC_DIR} đang là bản module cũ (thiếu get_metadata.get_entry_discovery_batch). "
        "Trên Colab: push bản mới nhất của src/ lên GitHub, xoá thư mục repo đã clone "
        f"(!rm -rf {REPO_ROOT}) rồi Restart session và chạy lại."
    )
if Path(utils.DATA_DIR).resolve() != DATA_DIR.resolve():
    raise RuntimeError(f"utils.DATA_DIR ({utils.DATA_DIR}) khác DATA_DIR của notebook ({DATA_DIR}).")
# Module đọc key lúc import; gán lại phòng khi module đã được import trước khi có key
if os.environ.get("S2_API_KEY"):
    ger._SESSION.headers["x-api-key"] = os.environ["S2_API_KEY"]


def setup_logging(log_file: Path) -> None:
    """Các module đều gọi basicConfig lúc import: thay bằng log INFO ra file, notebook chỉ hiện WARNING."""
    root = logging.getLogger()
    for h in list(root.handlers):
        root.removeHandler(h)
        h.close()
    fmt = logging.Formatter("%(asctime)s | %(threadName)-12s | %(name)s | %(levelname)s | %(message)s")
    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(fmt)
    console = logging.StreamHandler(sys.stdout)
    console.setLevel(logging.WARNING)
    console.setFormatter(fmt)
    root.addHandler(file_handler)
    root.addHandler(console)
    root.setLevel(logging.INFO)
    for noisy in ("arxiv", "urllib3", "botocore", "boto3", "s3transfer"):
        logging.getLogger(noisy).setLevel(logging.WARNING)


setup_logging(SCRAPER_LOG)
log = logging.getLogger("scraper_nb")
print("Log chi tiết:", SCRAPER_LOG)

# Phần II — Công cụ

Phần này **chỉ định nghĩa hàm**, không gọi mạng và không ghi dữ liệu.

- Ghi log và đọc trạng thái: tiến độ luôn được dựng lại từ các file state trên đĩa.
- §5 Đo đạc: `RunContext.timed`, `ResourceMonitor`.
- Adapter: lớp duy nhất gọi module của nhóm. Mọi exception của module được quy về hai loại:

| Loại | Ý nghĩa | Xử lý |
|---|---|---|
| `PaperError(status)` | lỗi riêng của một bài (`not_found`, `failed`) | ghi lại, đi tiếp; `failed` được thử lại ở phiên sau |
| `ServiceStop` | dịch vụ từ chối (403, hết quota S2) | dừng **làn đó**, làn còn lại chạy tiếp |
| `s3s.S3Unavailable` | thiếu AWS key / không có quyền đọc bucket | dừng làn S3, việc của nó trả về làn export |
| `ServiceStop(retryable=True)` | arXiv vẫn trả 429 / 5xx sau khi module tự retry | làn arXiv nghỉ `COOLDOWN_SEC` rồi thử lại **cùng request** (một bài hoặc một batch metadata), tối đa `MAX_COOLDOWNS` lần; quá số lần đó mới dừng làn |

In [ ]:
# Ghi log & đọc trạng thái
import csv
import threading
from collections import Counter, defaultdict
from contextlib import contextmanager

import numpy as np
import pandas as pd

_io_lock = threading.Lock()


def append_jsonl(path: Path, record: dict) -> None:
    """Ghi nối một dòng JSON và flush ngay (runtime bị ngắt cũng không mất dòng đã ghi)."""
    line = json.dumps(record, ensure_ascii=False)
    with _io_lock, open(path, "a", encoding="utf-8") as f:
        f.write(line + "\n")
        f.flush()


def read_jsonl(path: Path) -> list:
    """Đọc jsonl, bỏ qua dòng hỏng (dòng ghi dở khi runtime bị ngắt)."""
    if not path.exists():
        return []
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return out


def generate_ids(ranges: dict, labels: list, sample_n=None) -> pd.DataFrame:
    """DataFrame (arxiv_id, range_label); số thứ tự đệm 5 chữ số, vd 2608.04007."""
    rows = []
    for label in labels:
        yymm, start, end = ranges[label]
        stop = end if sample_n is None else min(end, start + sample_n - 1)
        rows += [(f"{yymm}.{n:05d}", label) for n in range(start, stop + 1)]
    df = pd.DataFrame(rows, columns=["arxiv_id", "range_label"])
    if df["arxiv_id"].duplicated().any():
        raise ValueError("Các range bị trùng ID.")
    return df


def read_metadata_file(arxiv_id: str):
    try:
        return json.loads((utils.get_paper_dir(arxiv_id) / "metadata.json").read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None


def read_refs_done() -> dict:
    """Bản ghi cuối của từng bài trong refs_done.jsonl (state của get_extracting_reference)."""
    last = {}
    for r in read_jsonl(STATE_DIR / "refs_done.jsonl"):
        if isinstance(r.get("id"), str):
            last[r["id"]] = r
    return last


def read_nb_status() -> dict:
    """{(stage, arxiv_id): bản ghi cuối} từ nb_status.jsonl."""
    return {(r["stage"], r["arxiv_id"]): r for r in read_jsonl(NB_STATUS_FILE)}


def source_status(entry) -> str:
    """Trạng thái source của một bài từ source_state.json."""
    if not entry:
        return "pending"
    if entry.get("completed"):
        # completed = mọi version có trạng thái kết thúc; "tex_bib" = đã extract xong
        return "done" if "tex_bib" in entry.get("sizes", {}) else "downloaded"
    return "partial" if entry.get("versions") else "pending"


def read_status(ids_df: pd.DataFrame) -> pd.DataFrame:
    """Trạng thái mỗi bài × mỗi phase, dựng lại hoàn toàn từ các file state trên đĩa."""
    nb = read_nb_status()
    refs = read_refs_done()
    src_papers = gfs.load_state().get("papers", {})
    on_disk = {e.name for e in os.scandir(DATA_DIR) if e.is_dir()}
    rows = []
    for aid in ids_df["arxiv_id"]:
        m = nb.get(("metadata", aid), {})
        meta_status, num_versions = m.get("status", "pending"), m.get("num_versions")
        if meta_status != "done" and utils.get_paper_dir(aid).name in on_disk:
            meta = read_metadata_file(aid)  # metadata.json có thể do chạy module trực tiếp mà có
            if meta and meta.get("arxiv_id") and meta.get("title"):
                meta_status, num_versions = "done", meta.get("num_versions")
        src = source_status(src_papers.get(aid))
        if src != "done" and nb.get(("source", aid), {}).get("status") == "failed":
            src = "failed"
        r = refs.get(aid)
        rows.append({
            "arxiv_id": aid,
            "metadata": meta_status,
            "num_versions": num_versions,
            "source": src,
            "refs": r["status"] if r else "pending",
            "n_refs": r.get("n_refs") if r else None,
        })
    return ids_df.merge(pd.DataFrame(rows), on="arxiv_id", how="left")


def build_todo(status_df: pd.DataFrame) -> dict:
    """Việc của phiên này: danh sách cho làn arXiv và làn S2."""
    retry = {"pending", "failed"} | ({"not_found"} if RETRY_NOT_FOUND else set())
    df = status_df.copy()
    df["need_metadata"] = df["metadata"].isin(retry) & RUN_METADATA
    df["need_source"] = (df["source"] != "done") & (df["need_metadata"] | (df["metadata"] == "done")) & RUN_SOURCE
    arxiv_todo = df.loc[df["need_metadata"] | df["need_source"],
                        ["arxiv_id", "need_metadata", "need_source", "num_versions"]]
    refs_todo = df.loc[df["refs"].isin(retry), "arxiv_id"].tolist() if RUN_REFS else []
    return {"arxiv": arxiv_todo.to_dict("records"), "refs": refs_todo}


def summarize_status(status_df: pd.DataFrame) -> pd.DataFrame:
    """Bảng đếm trạng thái theo range × phase."""
    parts = []
    for stage in ("metadata", "source", "refs"):
        t = pd.crosstab(status_df["range_label"], status_df[stage])
        t.columns = pd.MultiIndex.from_product([[stage], t.columns])
        parts.append(t)
    table = pd.concat(parts, axis=1).fillna(0).astype(int)
    table.loc["TOTAL"] = table.sum()
    return table


CONFIG_KEYS = ["RANGES_TO_RUN", "SAMPLE_N", "RUN_METADATA", "RUN_SOURCE", "RUN_REFS", "PARALLEL",
               "USE_S3", "S3_MAX_GB", "S3_LATEST_FOR_MULTI", "S3_MANIFEST_MAX_AGE_DAYS",
               "ARXIV_DELAY", "KEEP_ARCHIVE", "S2_BATCH_SIZE", "RETRY_NOT_FOUND", "MAX_CONSECUTIVE_ERRORS",
               "COOLDOWN_SEC", "MAX_COOLDOWNS", "METADATA_BATCH_SIZE",
               "MAX_RUN_HOURS", "MONITOR_EVERY", "DISK_EVERY", "SOURCE_STATE_FLUSH_EVERY", "PERSIST_TO_DRIVE"]


def run_config() -> dict:
    return {k: globals()[k] for k in CONFIG_KEYS}


def fmt_bytes(n) -> str:
    if n is None or (isinstance(n, float) and math.isnan(n)):
        return "-"
    for unit in ("B", "KB", "MB", "GB"):
        if abs(n) < 1024:
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"


def fmt_secs(s: float) -> str:
    h, rem = divmod(int(s), 3600)
    m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}"

## §5. Công cụ đo

- `RunContext.timed(stage, arxiv_id)`: ghi thời gian từng (bài, stage) vào `timings.jsonl`, kể cả khi lỗi.
- `ResourceMonitor`: thread nền ghi **RSS của tiến trình** (`psutil`, gồm cả tiến trình con) vào `monitor_<run_id>.csv` và **dung lượng `data/`** (quét như `du -sb`) vào `disk_<run_id>.csv`. Hai thread riêng để việc quét thư mục lớn không làm trễ mẫu RSS.
- `peak_rss_bytes()`: RSS đỉnh do hệ điều hành ghi nhận (`ru_maxrss` trên Linux) để đối chiếu với lấy mẫu.

In [ ]:
# §5 — Đo đạc
def dir_size(path: Path, exclude=frozenset()) -> int:
    """Tổng st_size mọi file dưới `path` (như `du -sb`); bỏ các thư mục con cấp 1 có tên trong `exclude`."""
    root, total, stack = str(path), 0, [str(path)]
    while stack:
        cur = stack.pop()
        try:
            with os.scandir(cur) as it:
                for e in it:
                    if e.is_dir(follow_symlinks=False):
                        if not (cur == root and e.name in exclude):
                            stack.append(e.path)
                    elif e.is_file(follow_symlinks=False):
                        total += e.stat(follow_symlinks=False).st_size
        except OSError:
            pass  # file/thư mục bị xoá trong lúc quét (archive tạm)
    return total


def peak_rss_bytes():
    """RSS đỉnh của kernel từ lúc khởi động (Linux: ru_maxrss; Windows: peak working set)."""
    try:
        import resource
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024
    except ImportError:
        return getattr(psutil.Process().memory_info(), "peak_wset", None)


class RunContext:
    """Trạng thái dùng chung của một phiên: run_id, cờ dừng, bộ đếm, ghi timings/status/events."""

    def __init__(self, run_id: str, max_hours=None):
        self.run_id = run_id
        self.stop_event = threading.Event()
        self.deadline = time.monotonic() + max_hours * 3600 if max_hours else None
        self.time_limit_hit = False
        self.counters = defaultdict(Counter)  # stage -> Counter(status)
        self.lane_stops = {}                  # lane -> lý do dừng sớm
        self.phase = {}                       # lane -> việc đang làm
        self.totals = {}                      # lane -> số việc của phiên
        self.progress = {}                    # lane -> số việc đã xử lý

    def should_stop(self) -> bool:
        if not self.stop_event.is_set() and self.deadline and time.monotonic() > self.deadline:
            self.time_limit_hit = True
            self.stop_event.set()
            self.event("all", "time_limit", f"Đạt MAX_RUN_HOURS={MAX_RUN_HOURS}")
        return self.stop_event.is_set()

    def phase_text(self) -> str:
        return " | ".join(f"{k}: {v}" for k, v in sorted(self.phase.items()))

    @contextmanager
    def timed(self, stage: str, arxiv_id: str, **extra):
        """Ghi {stage, arxiv_id, start, end, duration, status} vào timings.jsonl, kể cả khi có exception."""
        rec = {"run_id": self.run_id, "stage": stage, "arxiv_id": arxiv_id, "start": time.time(), **extra}
        t0, status = time.perf_counter(), "ok"
        try:
            yield
        except BaseException as e:
            status = getattr(e, "status", None) or f"error:{type(e).__name__}"
            raise
        finally:
            rec.update(end=time.time(), duration=round(time.perf_counter() - t0, 3), status=status)
            append_jsonl(TIMINGS_FILE, rec)

    def log_timing(self, rec: dict) -> None:
        append_jsonl(TIMINGS_FILE, {"run_id": self.run_id, **rec})

    def mark(self, stage: str, arxiv_id: str, status: str, **extra) -> None:
        """Ghi kết quả metadata/source vào nb_status.jsonl (dùng để resume)."""
        self.counters[stage][status] += 1
        append_jsonl(NB_STATUS_FILE, {"run_id": self.run_id, "stage": stage, "arxiv_id": arxiv_id,
                                      "status": status, "t": time.time(), **extra})

    def count(self, stage: str, status: str) -> None:
        self.counters[stage][status] += 1

    def event(self, lane: str, kind: str, message: str) -> None:
        log.warning("[%s] %s: %s", lane, kind, message)
        append_jsonl(EVENTS_FILE, {"run_id": self.run_id, "t": time.time(), "lane": lane,
                                   "kind": kind, "message": message})

    def stop_lane(self, lane: str, reason: str) -> None:
        self.lane_stops[lane] = reason
        self.event(lane, "lane_stop", reason)


class ResourceMonitor:
    """Lấy mẫu RSS mỗi `every` giây và dung lượng data/ mỗi `disk_every` giây, ghi CSV dạng append."""

    def __init__(self, run_id: str, every: float, disk_every: float, phase_fn=lambda: ""):
        self.proc = psutil.Process()
        self.every, self.disk_every, self.phase_fn = every, disk_every, phase_fn
        self.rss_path = LOG_DIR / f"monitor_{run_id}.csv"
        self.disk_path = LOG_DIR / f"disk_{run_id}.csv"
        self._stop = threading.Event()
        self._threads = []
        self.rss_last = self.rss_max = self.rss_sum = self.rss_n = 0
        self.disk_last = self.disk_max = None

    def rss_now(self) -> int:
        rss = self.proc.memory_info().rss
        for child in self.proc.children(recursive=True):
            try:
                rss += child.memory_info().rss
            except psutil.Error:
                pass
        return rss

    def _sample_rss(self) -> list:
        rss = self.rss_now()
        self.rss_last, self.rss_max = rss, max(self.rss_max, rss)
        self.rss_sum += rss
        self.rss_n += 1
        return [round(time.time(), 3), rss, self.phase_fn()]

    def _sample_disk(self) -> list:
        t0 = time.perf_counter()
        size = dir_size(DATA_DIR)
        self.disk_last, self.disk_max = size, max(self.disk_max or 0, size)
        return [round(time.time(), 3), size, round(time.perf_counter() - t0, 2)]

    def _loop(self, path: Path, header: list, interval: float, sample) -> None:
        new = not path.exists()
        with open(path, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            if new:
                w.writerow(header)
            stopping = False
            while not stopping:
                w.writerow(sample())
                f.flush()
                stopping = self._stop.wait(interval)
            w.writerow(sample())  # mẫu cuối lúc dừng
            f.flush()

    def start(self) -> None:
        self._stop.clear()
        self._threads = [
            threading.Thread(target=self._loop, name="monitor-rss", daemon=True,
                             args=(self.rss_path, ["t", "rss_bytes", "phase"], self.every, self._sample_rss)),
            threading.Thread(target=self._loop, name="monitor-disk", daemon=True,
                             args=(self.disk_path, ["t", "data_dir_bytes", "scan_sec"], self.disk_every, self._sample_disk)),
        ]
        for t in self._threads:
            t.start()

    def stop(self) -> None:
        self._stop.set()
        for t in self._threads:
            t.join()

    def summary(self) -> dict:
        return {
            "rss_max_bytes": self.rss_max,
            "rss_mean_bytes": round(self.rss_sum / self.rss_n) if self.rss_n else None,
            "rss_samples": self.rss_n,
            "disk_max_bytes": self.disk_max,
            "disk_last_bytes": self.disk_last,
        }

## Adapter

- `gfs2.Pacer`: một nhịp chung cho mọi request arXiv của làn (API metadata qua `arxiv.py` và `/src/`): request sau bắt đầu sau request trước ít nhất `ARXIV_DELAY` giây. Với lời gọi `arxiv.py` (tự gửi request bên trong), khoảng cách tính lại từ lúc lời gọi trả về.
- `SourceStateCache`: `source_state.json` giữ trong RAM, ghi xuống đĩa tối đa mỗi `SOURCE_STATE_FLUSH_EVERY` giây và một lần khi kết thúc phiên (với 25.000 bài, file này cỡ vài chục MB; ghi lại sau mỗi version rất tốn trên Drive). Hai làn source ghi vào cùng state nên dùng chung khoá `gfs2.STATE_LOCK`. Nếu runtime bị ngắt đột ngột thì mất tối đa `SOURCE_STATE_FLUSH_EVERY` giây tiến độ; những version đó chỉ bị tải lại.

In [ ]:
# Adapter — lớp duy nhất gọi module của nhóm
class PaperError(Exception):
    """Lỗi riêng của một bài: ghi lại rồi đi tiếp. status: 'not_found' | 'failed'."""

    def __init__(self, status: str, message: str = ""):
        super().__init__(message or status)
        self.status = status


class ServiceStop(Exception):
    """Dịch vụ từ chối. retryable=False (403 / hết quota): dừng cả làn, phiên sau chạy tiếp.
    retryable=True (429 / 5xx kéo dài): làn arXiv nghỉ COOLDOWN_SEC rồi thử lại cùng request."""
    status = "service_stop"

    def __init__(self, message: str = "", retryable: bool = False):
        super().__init__(message)
        self.retryable = retryable


# Giữ bản gốc một lần duy nhất, để chạy lại ô này không bọc chồng lên bản đã bọc
if not hasattr(gfs, "_nb_original_state_io"):
    gfs._nb_original_state_io = (gfs.load_state, gfs.save_state)
_ORIG_LOAD_STATE, _ORIG_SAVE_STATE = gfs._nb_original_state_io


class SourceStateCache:
    """source_state.json giữ trong RAM, ghi xuống đĩa tối đa mỗi `flush_every` giây."""

    def __init__(self, flush_every: float):
        self.flush_every = flush_every
        self.state = None
        self.dirty = False
        self.last_flush = time.monotonic()
        self.lock = gfs2.STATE_LOCK  # chung với hai làn source: không ghi file giữa lúc state đang đổi

    def load(self) -> dict:
        with self.lock:
            if self.state is None:
                self.state = _ORIG_LOAD_STATE()
            return self.state

    def save(self, state: dict) -> None:
        with self.lock:
            self.state, self.dirty = state, True
            if time.monotonic() - self.last_flush >= self.flush_every:
                self.flush()

    def flush(self) -> None:
        with self.lock:
            if self.dirty and self.state is not None:
                _ORIG_SAVE_STATE(self.state)
                self.dirty = False
            self.last_flush = time.monotonic()


# gfs2 gọi gfs.load_state / gfs.save_state lúc chạy, nên thay ở gfs là đủ
if SOURCE_STATE_FLUSH_EVERY > 0:
    SOURCE_STATE = SourceStateCache(SOURCE_STATE_FLUSH_EVERY)
    gfs.load_state, gfs.save_state = SOURCE_STATE.load, SOURCE_STATE.save
else:
    SOURCE_STATE = None
    gfs.load_state, gfs.save_state = _ORIG_LOAD_STATE, _ORIG_SAVE_STATE

SRC_SESSION = requests.Session()  # giữ kết nối keep-alive tới export.arxiv.org


def arxiv_api_error(e: arxiv.HTTPError) -> Exception:
    """Lỗi HTTP của arXiv API (module đã tự retry; tới đây là lỗi thật) -> ServiceStop / PaperError."""
    status = getattr(e, "status", None)
    if status == 403:
        return ServiceStop(f"arXiv API HTTP {status}")
    if status == 429 or (status or 0) >= 500:
        return ServiceStop(f"arXiv API HTTP {status}", retryable=True)
    return PaperError("failed", f"arXiv API HTTP {status}")


def fetch_metadata(arxiv_id: str) -> dict:
    """Phase A (từng bài): get_entry_discovery + save_metadata."""
    try:
        meta = gm.get_entry_discovery(arxiv_id)
    except ValueError as e:  # module báo không tìm thấy bài
        raise PaperError("not_found", str(e)) from e
    except arxiv.HTTPError as e:
        raise arxiv_api_error(e) from e
    except Exception as e:
        raise PaperError("failed", repr(e)) from e
    gm.save_metadata(arxiv_id, meta)
    return meta


def fetch_metadata_batch(ids: list) -> dict:
    """Phase A (batch): get_entry_discovery_batch + save_metadata. Trả {arxiv_id: metadata | None (not_found)}."""
    try:
        metas = gm.get_entry_discovery_batch(ids)
    except arxiv.HTTPError as e:
        raise arxiv_api_error(e) from e
    except Exception as e:
        raise PaperError("failed", repr(e)) from e
    for aid, meta in metas.items():
        if meta:
            gm.save_metadata(aid, meta)
    return metas


def fetch_source(arxiv_id: str, num_versions: int, versions, pacer: gfs2.Pacer) -> dict:
    """Phase B: tải qua export các version trong `versions` (None = mọi version) còn thiếu, chỉ giữ .tex/.bib."""
    try:
        entry = gfs2.scrape_paper(arxiv_id, num_versions, versions, pacer=pacer, session=SRC_SESSION,
                                  keep_archive=KEEP_ARCHIVE)
    except gfs.BlockedError as e:  # 403
        raise ServiceStop(str(e)) from e
    except gfs.RateLimitError as e:  # vẫn 429 / 5xx sau khi module tự chờ
        raise ServiceStop(str(e), retryable=True) from e
    except Exception as e:  # DownloadError (mạng, archive hỏng) và lỗi khác: thử lại ở phiên sau
        raise PaperError("failed", repr(e)) from e
    wanted = range(1, num_versions + 1) if versions is None else versions
    if gfs2.versions_left(entry, arxiv_id, wanted):
        raise PaperError("failed", "chưa tải đủ các version cần")
    return entry


def open_s3():
    """Làn S3: client + kiểm tra quyền + manifest. Thiếu key / không có quyền -> s3s.S3Unavailable."""
    client = s3s.make_client()
    s3s.check_access(client)
    return client, s3s.ChunkIndex(s3s.load_manifest(client, S3_MANIFEST_MAX_AGE_DAYS))


def fetch_refs_batch(ids: list) -> dict:
    """Phase C: một request batch; lưu references.json + refs_done.jsonl. Trả {arxiv_id: status}."""
    try:
        results = ger._fetch_batch(ids)
    except ger.QuotaExhausted as e:
        raise ServiceStop(str(e)) from e
    except Exception as e:
        raise PaperError("failed", repr(e)) from e
    out = {}
    for pid in ids:
        refs = results.get(pid)
        if refs is None:
            ger.mark_done(pid, "not_found", 0)
            out[pid] = "not_found"
        elif not refs:
            ger.mark_done(pid, "no_refs", 0)
            out[pid] = "no_refs"
        else:
            ger.save_extracting_references(pid, refs)
            ger.mark_done(pid, "ok", len(refs))
            out[pid] = "ok"
    return out

# Phần III — Cào dữ liệu

## §4. Danh sách ID và tình trạng hiện tại

Bảng dưới được dựng lại từ các file state, cho biết lần chạy này sẽ tiếp tục từ đâu.

| Phase | Trạng thái |
|---|---|
| `metadata` | `pending`, `done`, `not_found` (arXiv không có bài này), `failed` (sẽ thử lại) |
| `source` | `pending`, `partial` (tải dở), `downloaded` (chưa extract), `done`, `failed` |
| `refs` | `pending`, `ok`, `no_refs`, `not_found` (Semantic Scholar chưa có bài) |

In [ ]:
# §4 — ID của các range được chọn + tình trạng từ state
ids_df = generate_ids(RANGES, RANGES_TO_RUN, SAMPLE_N)
status_df = read_status(ids_df)
todo_preview = build_todo(status_df)

print(f"{len(ids_df)} ID | còn việc: làn arXiv {len(todo_preview['arxiv'])} bài, làn S2 {len(todo_preview['refs'])} bài")
display(summarize_status(status_df))

## §6–§7. Làn arXiv: Phase A (entry discovery), kế hoạch, Phase B (export)

Một thread, tuần tự, một kết nối, một nhịp `ARXIV_DELAY` chung cho mọi request:

1. **Phase A theo batch** cho mọi bài còn thiếu metadata: mỗi request `get_entry_discovery_batch` lấy bản mới nhất của tới `METADATA_BATCH_SIZE` bài, cộng các request gom `v1..vN` của bài nhiều version (để có ngày từng version). Bài không có trong kết quả là `not_found`. Nếu cả batch lỗi thì từng bài tự lấy metadata riêng bằng `get_entry_discovery`.
2. **Kế hoạch** (`make_plan`): chờ làn S3 mở bucket, rồi chia việc source như ở §0. Dòng sự kiện `plan` in ra số bài, số chunk, số GB tối đa sẽ đọc và chi phí ước tính.
3. **Phase B** cho phần của export: tải các version còn thiếu qua `/src/`, chỉ giữ `.tex` / `.bib`. Version chỉ có PDF thì giữ thư mục rỗng.
4. **Việc làn S3 trả lại** (bài không có trong chunk, archive hỏng, S3 bị từ chối): làn tiếp tục nhận từ hàng đợi cho tới khi làn S3 dừng.

Nếu arXiv vẫn trả 429 / 5xx sau khi module tự chờ, làn **nghỉ `COOLDOWN_SEC` giây rồi thử lại cùng request**, tối đa `MAX_COOLDOWNS` lần (Interrupt hoặc `MAX_RUN_HOURS` cắt ngang được lúc nghỉ). Gặp 403 thì dừng làn ngay. Mỗi lần nghỉ ghi một sự kiện `cooldown` vào `events.jsonl`.

**Tổng thời gian entry discovery** = tổng thời lượng các stage `metadata` trong `timings.jsonl` (thời gian batch chia đều cho các bài, ghi `amortized: true`).

In [ ]:
# §6–§7 — Làn arXiv và kế hoạch chia việc với làn S3
import queue


class LaneInterrupted(Exception):
    """Interrupt / MAX_RUN_HOURS trong lúc đang nghỉ cooldown."""


def sleep_unless_stopped(ctx: RunContext, seconds: float) -> bool:
    """Nghỉ `seconds` giây; dậy sớm khi Interrupt hoặc đạt MAX_RUN_HOURS. True nếu nghỉ đủ."""
    end = time.monotonic() + seconds
    while not ctx.should_stop():
        left = end - time.monotonic()
        if left <= 0:
            return True
        ctx.stop_event.wait(min(left, 30.0))  # thức dậy định kỳ để should_stop() kiểm tra deadline
    return False


def wait_event(ctx: RunContext, event: threading.Event) -> bool:
    """Chờ `event` của làn khác; False nếu bị Interrupt / hết giờ trước."""
    while not event.wait(5.0):
        if ctx.should_stop():
            return False
    return True


def with_cooldown(ctx: RunContext, label: str, fn):
    """Gọi fn(). arXiv 429/5xx kéo dài (ServiceStop retryable) -> nghỉ COOLDOWN_SEC rồi gọi lại, tối đa
    MAX_COOLDOWNS lần; hết lượt hoặc 403 -> ServiceStop có kèm `label`. Bị dừng lúc nghỉ -> LaneInterrupted."""
    cooldowns = 0
    while True:
        try:
            return fn()
        except ServiceStop as e:
            if not e.retryable or cooldowns >= MAX_COOLDOWNS:
                raise ServiceStop(f"{label}: {e}" + (f" (sau {cooldowns} lần cooldown)" if cooldowns else "")) from e
            cooldowns += 1
            ctx.count("cooldown", "arxiv")
            ctx.event("arxiv", "cooldown",
                      f"{label}: {e}; nghỉ {COOLDOWN_SEC:.0f}s rồi thử lại ({cooldowns}/{MAX_COOLDOWNS})")
            ctx.phase["arxiv"] = f"cooldown {label} ({cooldowns}/{MAX_COOLDOWNS})"
            if not sleep_unless_stopped(ctx, COOLDOWN_SEC):
                raise LaneInterrupted() from e


def timed(ctx: RunContext, stage: str, aid: str, fn, *args, **extra):
    with ctx.timed(stage, aid, **extra):
        return fn(*args)


class HybridPlan:
    """Chỗ hẹn giữa làn arXiv và làn S3 trong một phiên."""

    def __init__(self, use_s3: bool):
        self.s3_ready = threading.Event()     # làn S3 đã mở bucket (index) hoặc đã bỏ cuộc (index = None)
        self.planned = threading.Event()      # làn arXiv đã chia việc (s3_plan)
        self.s3_finished = threading.Event()  # làn S3 đã dừng: hàng đợi fallback không nhận thêm việc
        self.index = None                     # s3s.ChunkIndex
        self.chunk_state = s3s.load_chunk_state()
        self.s3_plan = s3s.Plan()
        self.num_versions = {}                # arxiv_id -> num_versions, cho các bài giao cho S3
        self.fallback = queue.Queue()         # việc S3 trả lại cho export: {arxiv_id, num_versions, versions}
        if not use_s3:
            self.s3_ready.set()
            self.s3_finished.set()

    def give_back(self, arxiv_id: str, version: int) -> None:
        self.fallback.put({"arxiv_id": arxiv_id, "num_versions": self.num_versions[arxiv_id], "versions": [version]})


def num_versions_of(item: dict):
    """num_versions của một việc (từ Phase A, từ trạng thái, hoặc đọc lại metadata.json); None nếu không có."""
    nv = item.get("num_versions")
    if nv is None or pd.isna(nv):
        nv = (read_metadata_file(item["arxiv_id"]) or {}).get("num_versions")
    return int(nv) if nv else None


def s3_has_latest(chunk, arxiv_id: str) -> bool:
    """Chunk đóng gói sau ngày của version mới nhất (cách hơn 1 ngày) -> bản trong chunk có thể là vN."""
    meta = read_metadata_file(arxiv_id) or {}
    dates = [d for d in [meta.get("submission_date"), *(meta.get("revised_dates") or [])] if d]
    if not dates or not chunk.timestamp:
        return False
    return chunk.timestamp[:10] > (pd.Timestamp(max(dates)) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")


def make_plan(ctx: RunContext, plan: HybridPlan, items: list) -> list:
    """Chia việc source của phiên; trả danh sách việc cho export (phần của S3 nằm ở plan.s3_plan).

    - Bài chỉ có v1, nằm trong một chunk S3 -> làn S3.
    - S3_LATEST_FOR_MULTI: vN của bài nhiều version -> làn S3, v1..vN-1 -> export.
    - Còn lại (bài nhiều version, bài chưa lên S3, chunk vượt S3_MAX_GB) -> export mọi version còn thiếu.
    """
    ctx.phase["arxiv"] = "chờ làn S3 mở bucket"
    if not wait_event(ctx, plan.s3_ready):
        raise LaneInterrupted()
    papers = gfs.load_state()["papers"]
    targets = {}
    for it in items:
        aid, nv = it["arxiv_id"], it["num_versions"]
        it["versions"] = None
        if plan.index is None:
            continue
        if nv == 1:
            version = 1
        elif S3_LATEST_FOR_MULTI and (chunk := plan.index.find(aid)) is not None and s3_has_latest(chunk, aid):
            version = nv
        else:
            continue
        if gfs2.versions_left(papers.get(aid, {"versions": {}}), aid, [version]):
            targets[aid] = version
            plan.num_versions[aid] = nv
    if plan.index is not None:
        plan.s3_plan = s3s.plan(plan.index, targets, plan.chunk_state, S3_MAX_GB)
    on_s3 = {aid: v for wanted in plan.s3_plan.work.values() for aid, v in wanted.items()}
    export = []
    for it in items:
        if it["arxiv_id"] not in on_s3:
            export.append(it)
        elif it["num_versions"] > 1:  # vN từ S3, các version trước qua export
            it["versions"] = list(range(1, it["num_versions"]))
            export.append(it)
    p = plan.s3_plan
    ctx.event("arxiv", "plan", f"S3: {p.n_papers} bài trong {len(p.work)} chunk (đọc tối đa {p.gb_to_read:.1f} GB, "
                               f"~{p.cost_usd:.2f} USD); export: {len(export)} bài")
    return export


def prefetch_metadata(ctx: RunContext, pacer: gfs2.Pacer, chunk: list) -> None:
    """Phase A theo batch cho các bài trong `chunk` (sửa item tại chỗ).
    Batch lỗi (PaperError): giữ nguyên need_metadata để từng bài tự lấy metadata riêng."""
    ids = [it["arxiv_id"] for it in chunk]
    label = f"batch {ids[0]}..{ids[-1]}"
    ctx.phase["arxiv"] = f"A {label} ({len(ids)} bài)"

    def attempt():
        t0 = time.perf_counter()
        return pacer.call(fetch_metadata_batch, ids), time.perf_counter() - t0

    t_start = time.time()
    try:
        metas, duration = with_cooldown(ctx, label, attempt)
    except PaperError as e:
        ctx.event("arxiv", "metadata_batch_failed", f"{label}: {e}; chuyển sang lấy từng bài")
        return
    t_end = time.time()
    for it in chunk:
        aid, meta = it["arxiv_id"], metas.get(it["arxiv_id"])
        ctx.log_timing({"stage": "metadata", "arxiv_id": aid, "start": t_start, "end": t_end,
                        "duration": round(duration / len(chunk), 3), "status": "ok" if meta else "not_found",
                        "batch_size": len(chunk), "amortized": True})
        it["need_metadata"] = False
        if meta:
            it["num_versions"] = meta["num_versions"]
            ctx.mark("metadata", aid, "done", num_versions=meta["num_versions"])
        else:  # arXiv bỏ qua id không tồn tại
            it["need_source"] = False
            ctx.mark("metadata", aid, "not_found", error="không có trong kết quả batch")


def arxiv_lane(ctx: RunContext, todo: list, plan: HybridPlan) -> None:
    pacer = gfs2.Pacer(ARXIV_DELAY)  # một nhịp chung cho API metadata và /src/
    consecutive_errors = 0

    def attempt(stage: str, aid: str, fn) -> bool:
        """Một việc của một bài; lỗi riêng của bài được ghi lại. False nếu làn phải dừng."""
        nonlocal consecutive_errors
        try:
            fn()
            consecutive_errors = 0
        except PaperError as e:
            ctx.mark(stage, aid, e.status, error=str(e)[:500])
            if e.status != "failed":  # not_found là kết quả hợp lệ
                consecutive_errors = 0
                return True
            consecutive_errors += 1
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                ctx.stop_lane("arxiv", f"{consecutive_errors} lỗi liên tiếp, lỗi cuối ({aid}): {e}")
                return False
        return True

    def metadata_one(it: dict) -> None:
        aid = it["arxiv_id"]
        ctx.phase["arxiv"] = f"A {aid}"
        meta = with_cooldown(ctx, aid, lambda: timed(ctx, "metadata", aid, pacer.call, fetch_metadata, aid))
        it["need_metadata"], it["num_versions"] = False, meta["num_versions"]
        ctx.mark("metadata", aid, "done", num_versions=meta["num_versions"])

    def source_one(it: dict) -> None:
        aid, nv, versions = it["arxiv_id"], it["num_versions"], it.get("versions")
        ctx.phase["arxiv"] = f"B {aid} ({nv} version" + ("" if versions is None else f", tải {versions}") + ")"
        entry = with_cooldown(ctx, aid, lambda: timed(ctx, "source", aid, fetch_source, aid, nv, versions, pacer,
                                                      num_versions=nv, via="export"))
        if entry.get("completed"):
            ctx.mark("source", aid, "done", num_versions=nv)

    try:
        # 1. Phase A
        need = [it for it in todo if it["need_metadata"]]
        ctx.totals["arxiv A"], ctx.progress["arxiv A"] = len(need), 0
        for start in range(0, len(need), METADATA_BATCH_SIZE):
            if ctx.should_stop():
                return
            prefetch_metadata(ctx, pacer, need[start:start + METADATA_BATCH_SIZE])
            ctx.progress["arxiv A"] = min(start + METADATA_BATCH_SIZE, len(need))
        for it in need:
            if it["need_metadata"]:  # batch của nó bị lỗi
                if ctx.should_stop():
                    return
                if not attempt("metadata", it["arxiv_id"], lambda: metadata_one(it)):
                    return

        # 2. Kế hoạch
        ready = []
        for it in todo:
            if it["need_source"] and not it["need_metadata"]:
                it["num_versions"] = num_versions_of(it)
                if it["num_versions"]:
                    ready.append(it)
                else:
                    ctx.mark("source", it["arxiv_id"], "failed", error="không đọc được num_versions")
        export = make_plan(ctx, plan, ready)
        plan.planned.set()

        # 3. Phase B qua export, rồi 4. việc làn S3 trả lại
        ctx.totals["arxiv B"], ctx.progress["arxiv B"] = len(export), 0
        for it in export:
            if ctx.should_stop():
                return
            if not attempt("source", it["arxiv_id"], lambda: source_one(it)):
                return
            ctx.progress["arxiv B"] += 1
        while not (plan.s3_finished.is_set() and plan.fallback.empty()):
            if ctx.should_stop():
                return
            ctx.phase["arxiv"] = "chờ việc làn S3 trả lại"
            try:
                it = plan.fallback.get(timeout=5.0)
            except queue.Empty:
                continue
            ctx.totals["arxiv B"] += 1
            if not attempt("source", it["arxiv_id"], lambda: source_one(it)):
                return
            ctx.progress["arxiv B"] += 1
    except ServiceStop as e:  # 403, hoặc vẫn 429/5xx sau MAX_COOLDOWNS lần nghỉ
        ctx.stop_lane("arxiv", str(e))
    except LaneInterrupted:  # Interrupt / hết giờ lúc đang nghỉ: không coi là dịch vụ từ chối
        pass
    finally:
        plan.planned.set()  # làn S3 không chờ mãi khi làn arXiv dừng sớm
        ctx.phase["arxiv"] = "dừng"

## §7b. Làn S3: full-text source từ bucket bulk

1. Mở bucket: kiểm tra key và quyền (một HEAD), đọc manifest (lưu trong `data/.state`, tải lại sau `S3_MANIFEST_MAX_AGE_DAYS` ngày). Thất bại thì dừng làn; làn arXiv sẽ giao mọi việc cho export.
2. Chờ kế hoạch của làn arXiv, rồi đọc lần lượt từng chunk được giao (`s3s.process_chunk`). Mỗi bài cần lấy được lưu ngay bằng `gfs2.store_fetched`, giống hệt một version tải từ export (`via = s3:<chunk>`).
3. Bài không có trong chunk, archive hỏng, hoặc mọi bài còn lại khi S3 từ chối giữa chừng: **trả về hàng đợi của làn export**.

Thời gian mỗi bài = thời gian đọc chunk / số bài lấy được từ chunk (ghi `amortized: true`, `via: s3` trong `timings.jsonl`). Dung lượng đã đọc (phần nhóm trả tiền) nằm trong `data/.state/s3_chunks.json` và được cộng vào §12.

In [ ]:
# §7b — Làn S3
def s3_lane(ctx: RunContext, plan: HybridPlan) -> None:
    ctx.phase["s3"] = "mở bucket"
    try:
        try:
            client, plan.index = open_s3()
        except Exception as e:  # thiếu key, không có quyền, lỗi mạng
            ctx.stop_lane("s3", f"không dùng được S3 ({e}); mọi version đi qua export")
            return
        finally:
            plan.s3_ready.set()
        if not wait_event(ctx, plan.planned):
            return
        work = plan.s3_plan.work
        ctx.totals["s3"], ctx.progress["s3"] = plan.s3_plan.n_papers, 0
        state = gfs.load_state()

        def unfinished(wanted: dict) -> dict:
            return {a: v for a, v in wanted.items()
                    if gfs2.versions_left(state["papers"].get(a, {"versions": {}}), a, [v])}

        names = list(work)
        for i, name in enumerate(names):
            if ctx.should_stop():
                return
            chunk, wanted = plan.index.by_name[name], work[name]
            ctx.phase["s3"] = (f"{name.rsplit('/', 1)[-1]} ({i + 1}/{len(names)} chunk, {len(wanted)} bài, "
                               f"{chunk.size / 1e9:.2f} GB)")

            def store(aid: str, version: int, data: bytes, name=name) -> None:
                gfs2.store_fetched(aid, plan.num_versions[aid], version, gfs2.Fetched(data, len(data)),
                                   via=f"s3:{name}", keep_archive=KEEP_ARCHIVE, state=state)

            t_start = time.time()
            try:
                res = s3s.process_chunk(client, chunk, wanted, store, plan.chunk_state, ctx.should_stop)
            except s3s.S3Unavailable as e:  # key bị thu hồi / hết quyền giữa chừng
                ctx.stop_lane("s3", f"{name}: {e}; trả việc còn lại cho export")
                for later in names[i:]:
                    for aid, v in unfinished(work[later]).items():
                        plan.give_back(aid, v)
                return
            except Exception as e:  # stream hỏng quá s3s.STREAM_RETRIES lần
                ctx.event("s3", "chunk_failed", f"{name}: {e!r}; trả {len(wanted)} bài cho export")
                for aid, v in unfinished(wanted).items():
                    plan.give_back(aid, v)
                continue
            t_end = time.time()
            for aid in res.stored:
                ctx.log_timing({"stage": "source", "arxiv_id": aid, "start": t_start, "end": t_end,
                                "duration": round(res.seconds / len(res.stored), 3), "status": "ok",
                                "via": "s3", "chunk": name, "amortized": True})
                if state["papers"][aid].get("completed"):
                    ctx.mark("source", aid, "done", num_versions=plan.num_versions[aid], via="s3")
            for aid in res.missing + res.failed:
                plan.give_back(aid, wanted[aid])
            ctx.counters["s3"]["stored"] += len(res.stored)
            ctx.counters["s3"]["to_export"] += len(res.missing) + len(res.failed)
            ctx.counters["s3"]["MB_read"] += round(res.bytes_read / 1e6)
            ctx.progress["s3"] += len(res.stored) + len(res.missing) + len(res.failed)
    finally:
        plan.s3_ready.set()
        plan.s3_finished.set()
        ctx.phase["s3"] = "dừng"

## §8. Làn Semantic Scholar: Phase C (references)

Mỗi request batch lấy reference của `S2_BATCH_SIZE` bài (endpoint `/paper/batch` của module). Mọi reference được lưu trong `references.json` của bài, kết quả từng bài ghi vào `refs_done.jsonl`.

Thời gian mỗi bài = thời gian của batch / số bài trong batch (ghi với `amortized: true` trong `timings.jsonl`).

In [ ]:
# §8 — Làn Semantic Scholar
def s2_lane(ctx: RunContext, ids: list) -> None:
    consecutive_errors = 0
    n_batches = math.ceil(len(ids) / S2_BATCH_SIZE)
    ctx.totals["s2"] = len(ids)
    for b, start in enumerate(range(0, len(ids), S2_BATCH_SIZE), 1):
        if ctx.should_stop():
            break
        batch = ids[start:start + S2_BATCH_SIZE]
        ctx.phase["s2"] = f"C batch {b}/{n_batches}"
        t_start, t0 = time.time(), time.perf_counter()
        stop_reason = None
        try:
            results = fetch_refs_batch(batch)
            consecutive_errors = 0
        except ServiceStop as e:
            results, stop_reason = {}, str(e)
        except PaperError as e:
            results = {pid: "failed" for pid in batch}
            consecutive_errors += 1
            ctx.event("s2", "batch_failed", f"batch {b} ({batch[0]}...): {e}")
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                stop_reason = f"{consecutive_errors} batch lỗi liên tiếp"
        duration, t_end = time.perf_counter() - t0, time.time()
        for pid, status in results.items():
            ctx.count("refs", status)
            ctx.log_timing({"stage": "refs", "arxiv_id": pid, "start": t_start, "end": t_end,
                            "duration": round(duration / len(batch), 3), "status": status,
                            "batch_size": len(batch), "amortized": True})
        ctx.progress["s2"] = start + len(batch)
        if stop_reason:
            ctx.stop_lane("s2", stop_reason)
            break
    ctx.phase["s2"] = "dừng"

## §9. Chạy

`run(ids_df)` là **một phiên**:

1. Đọc state, lập danh sách việc còn lại.
2. Bật monitor, chạy làn arXiv ‖ làn S3 ‖ làn S2 (`PARALLEL = False`: làn S2 chạy sau hai làn source, để so sánh).
3. In tiến độ mỗi `PROGRESS_EVERY` giây.
4. Kết thúc khi hết việc, khi dịch vụ từ chối (403, hết quota, hoặc arXiv vẫn 429 sau `MAX_COOLDOWNS` lần nghỉ), khi đạt `MAX_RUN_HOURS`, hoặc khi bấm **Interrupt** (Ctrl+C). Dù dừng vì lý do nào, notebook vẫn flush state và ghi dòng kết thúc vào `runs.jsonl`.
   - Khi Interrupt, notebook chờ tối đa `INTERRUPT_GRACE` giây cho các làn xong việc hiện tại. Nếu module đang tự chờ bên trong (có thể vài phút), làn đó tiếp tục chạy nền tới hết việc hiện tại rồi tự dừng; dữ liệu nó ghi vẫn hợp lệ.

Bị dừng thì **chạy lại ô dưới** (hoặc cả notebook ở phiên Colab mới) để tiếp tục.

In [ ]:
# §9 — Điều phối một phiên
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait

# Các làn của phiên trước (có thể còn chạy nền sau Interrupt); giữ qua các lần chạy lại ô này
ACTIVE_LANES = globals().get("ACTIVE_LANES", [])

def print_progress(ctx: RunContext, monitor: ResourceMonitor, t0: float) -> None:
    lanes = " | ".join(f"{lane} {ctx.progress.get(lane, 0)}/{n}" for lane, n in ctx.totals.items())
    print(f"[{time.strftime('%H:%M:%S')}] {fmt_secs(time.time() - t0)} | {lanes} | "
          f"RSS {fmt_bytes(monitor.rss_last)} | data/ {fmt_bytes(monitor.disk_last)}")
    for stage, c in ctx.counters.items():
        print(f"    {stage}: " + ", ".join(f"{k}={v}" for k, v in sorted(c.items())))


def run(ids_df: pd.DataFrame) -> dict:
    """Một phiên cào. Chạy lại để tiếp tục từ state."""
    leftover = [f for f in ACTIVE_LANES if not f.done()]
    if leftover:  # không bao giờ để hai làn arXiv chạy cùng lúc
        print(f"Đợi {len(leftover)} làn của phiên trước dừng hẳn...")
        wait(leftover)
    todo = build_todo(read_status(ids_df))
    if not todo["arxiv"] and not todo["refs"]:
        print("Không còn việc. Chuyển sang Phần IV.")
        return {"run_id": None, "stop_reason": "nothing_to_do"}

    run_id = time.strftime("%Y%m%d-%H%M%S")
    ctx = RunContext(run_id, MAX_RUN_HOURS)
    use_s3 = USE_S3 and RUN_SOURCE and bool(todo["arxiv"])
    plan = HybridPlan(use_s3)
    monitor = ResourceMonitor(run_id, MONITOR_EVERY, DISK_EVERY, ctx.phase_text)
    print(f"run_id={run_id} | làn arXiv: {len(todo['arxiv'])} bài | làn S3: {'bật' if use_s3 else 'tắt'} "
          f"| làn S2: {len(todo['refs'])} bài")
    append_jsonl(RUNS_FILE, {"run_id": run_id, "event": "start", "t": time.time(), "machine": machine_info(),
                             "config": run_config(), "todo": {k: len(v) for k, v in todo.items()}})
    monitor.start()
    t0, stop_reason = time.time(), "completed"
    executor = ThreadPoolExecutor(max_workers=3, thread_name_prefix="lane")
    futures, wait_lanes = [], True
    try:
        if todo["arxiv"]:
            futures.append(executor.submit(arxiv_lane, ctx, todo["arxiv"], plan))
        if use_s3:
            futures.append(executor.submit(s3_lane, ctx, plan))
        source_lanes = list(futures)
        s2_waiting = bool(todo["refs"])
        if s2_waiting and (PARALLEL or not source_lanes):
            futures.append(executor.submit(s2_lane, ctx, todo["refs"]))
            s2_waiting = False
        ACTIVE_LANES[:] = futures
        pending = set(futures)
        while pending:
            done, pending = wait(pending, timeout=PROGRESS_EVERY, return_when=FIRST_COMPLETED)
            for f in done:
                f.result()  # lỗi lập trình trong một làn -> dừng cả phiên
            if s2_waiting and all(f.done() for f in source_lanes) and not ctx.should_stop():
                s2 = executor.submit(s2_lane, ctx, todo["refs"])  # PARALLEL = False: S2 sau hai làn source
                futures.append(s2)
                pending.add(s2)
                s2_waiting = False
            print_progress(ctx, monitor, t0)
    except KeyboardInterrupt:
        stop_reason = "interrupted"
        ctx.stop_event.set()
        print(f"Đang dừng: chờ tối đa {INTERRUPT_GRACE:.0f}s để các làn xong việc hiện tại "
              "(bấm Interrupt lần nữa để thôi chờ)...")
        try:
            _, still_running = wait(futures, timeout=INTERRUPT_GRACE)
        except KeyboardInterrupt:
            still_running = [f for f in futures if not f.done()]
        if still_running:
            # Module đang tự retry bên trong (có thể vài phút): không chờ nữa; làn sẽ tự dừng sau việc hiện tại
            wait_lanes = False
            stop_reason = "interrupted (làn còn chạy nền tới hết việc hiện tại)"
    except BaseException as e:
        stop_reason = f"error: {e!r}"
        ctx.stop_event.set()
        raise
    finally:
        executor.shutdown(wait=wait_lanes)
        if SOURCE_STATE is not None:
            SOURCE_STATE.flush()
        monitor.stop()
        if stop_reason == "completed" and ctx.time_limit_hit:
            stop_reason = "time_limit"
        elif stop_reason == "completed" and ctx.lane_stops:
            stop_reason = "lane_stop"
        end_rec = {"run_id": run_id, "event": "end", "t": time.time(), "wall_sec": round(time.time() - t0, 2),
                   "stop_reason": stop_reason, "lane_stops": ctx.lane_stops,
                   "counters": {k: dict(v) for k, v in ctx.counters.items()},
                   "monitor": monitor.summary(), "peak_rss_kernel_bytes": peak_rss_bytes(),
                   "s3_bytes_read_total": s3s.bytes_read_total(plan.chunk_state)}
        append_jsonl(RUNS_FILE, end_rec)
        print(f"Kết thúc phiên {run_id}: {stop_reason} sau {fmt_secs(end_rec['wall_sec'])}")
        if ctx.lane_stops:
            print("  Làn dừng sớm:", ctx.lane_stops)
        print("  RSS max:", fmt_bytes(monitor.rss_max), "| data/ max:", fmt_bytes(monitor.disk_max))
    return end_rec

In [ ]:
# §9 — CHẠY (chạy lại ô này để tiếp tục sau khi bị dừng)
last_run = run(ids_df)

In [ ]:
# §9 — Tình trạng sau phiên
status_df = read_status(ids_df)
display(summarize_status(status_df))

# Phần IV — Phân tích và export

Phần này **chỉ đọc đĩa**. Chạy sau khi cào xong để chốt số liệu, hoặc chạy giữa chừng để xem tiến độ.

## §10. Bảng dữ liệu

- `papers`: 6 cột theo đề (`arxiv_id, title, authors, submission_date, revised_dates, venue`). Các cột thêm (`num_versions, categories, abstract`) nằm trong `papers_full` và trong `per_paper.csv`.
- `references_all`: **mọi** reference crawl được (đề yêu cầu giữ local).
- `references`: chỉ những reference có `ref_arxiv_id`, bỏ cặp `(arxiv_id, ref_arxiv_id)` trùng. Đây là bảng sẽ nộp lên Hugging Face.

In [ ]:
# §10 — Dựng bảng từ metadata.json / references.json
from concurrent.futures import ThreadPoolExecutor

from tqdm import tqdm  # thanh tiến độ dạng text: chạy được cả khi không có ipywidgets

PAPERS_COLUMNS = ["arxiv_id", "title", "authors", "submission_date", "revised_dates", "venue"]
PAPERS_EXTRA = ["num_versions", "categories", "abstract"]
REF_COLUMNS = ["arxiv_id", "ref_arxiv_id", "ref_title", "ref_authors", "ref_submission_date", "ref_s2_id"]


def read_json_many(paths: list, workers: int = 16) -> list:
    """Đọc nhiều file JSON song song (Drive chậm khi đọc tuần tự từng file nhỏ)."""
    def load(p):
        try:
            return json.loads(Path(p).read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            return None
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(tqdm(pool.map(load, paths), total=len(paths), leave=False))


status_df = read_status(ids_df)

meta_ids = status_df.loc[status_df["metadata"] == "done", "arxiv_id"].tolist()
metas = [m for m in read_json_many([utils.get_paper_dir(a) / "metadata.json" for a in meta_ids]) if m]
papers_full = pd.DataFrame(metas, columns=PAPERS_COLUMNS + PAPERS_EXTRA).merge(ids_df, on="arxiv_id", how="left")
papers = papers_full[PAPERS_COLUMNS].reset_index(drop=True)

ok_ids = status_df.loc[status_df["refs"] == "ok", "arxiv_id"].tolist()
ref_lists = read_json_many([utils.get_paper_dir(a) / "references.json" for a in ok_ids])
references_all = pd.DataFrame([r for refs in ref_lists if refs for r in refs], columns=REF_COLUMNS)
with_arxiv = references_all[references_all["ref_arxiv_id"].notna()]
references = with_arxiv.drop_duplicates(["arxiv_id", "ref_arxiv_id"]).reset_index(drop=True)
n_ref_duplicates = len(with_arxiv) - len(references)

print(f"papers: {len(papers)} | references_all: {len(references_all)} | references (có arXiv ID): {len(references)}")
display(papers.head(3))
display(references.head(3))

In [ ]:
# §10 — Kiểm tra schema
ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2}")


def is_iso(d) -> bool:
    return isinstance(d, str) and bool(ISO_DATE.match(d))


def validate_tables() -> pd.DataFrame:
    expected_refs = pd.to_numeric(status_df.loc[status_df["refs"] == "ok", "n_refs"], errors="coerce").sum()
    checks = [
        ("papers", "arxiv_id null", papers["arxiv_id"].isna().sum()),
        ("papers", "arxiv_id trùng", papers["arxiv_id"].duplicated().sum()),
        ("papers", "title rỗng", (papers["title"].fillna("").str.len() == 0).sum()),
        ("papers", "authors không phải list", (~papers["authors"].map(lambda a: isinstance(a, list))).sum()),
        ("papers", "submission_date không phải ISO", (~papers["submission_date"].map(is_iso)).sum()),
        ("papers", "revised_dates có ngày không phải ISO",
         papers["revised_dates"].map(lambda ds: not all(is_iso(d) for d in (ds or []))).sum()),
        ("references", "arxiv_id / ref_arxiv_id null", references[["arxiv_id", "ref_arxiv_id"]].isna().any(axis=1).sum()),
        ("references", "cặp (arxiv_id, ref_arxiv_id) trùng (đã bỏ)", n_ref_duplicates),
        ("references_all", "lệch giữa số dòng và tổng n_refs trong refs_done", abs(len(references_all) - int(expected_refs))),
    ]
    table = pd.DataFrame(checks, columns=["bảng", "kiểm tra", "số vi phạm"])
    table["số vi phạm"] = table["số vi phạm"].astype(int)
    return table


display(validate_tables())
old_style = references["ref_arxiv_id"].astype(str).str.contains("/").sum()
print(f"ref_arxiv_id kiểu cũ (archive/YYMMNNN, vd hep-th/9901001), giữ nguyên: {old_style}")

In [ ]:
# §10 — Ghi parquet và đọc lại để kiểm tra
TABLE_FILES = {
    "papers": (papers, TABLE_DIR / "papers.parquet"),
    "references": (references, TABLE_DIR / "references.parquet"),
    "references_all": (references_all, TABLE_DIR / "references_all.parquet"),
}
for name, (df, path) in TABLE_FILES.items():
    df.to_parquet(path, index=False)
    back = pd.read_parquet(path)
    assert len(back) == len(df) and list(back.columns) == list(df.columns), name
    print(f"{path.name:24s} {len(back):>9} dòng  {fmt_bytes(path.stat().st_size):>10}")

### Kiểm tra corpus: chỉ còn `.tex` / `.bib`

Đề yêu cầu bỏ hình bằng cách **extract chọn lọc**: mỗi thư mục version chỉ được chứa `.tex` / `.bib`. Thư mục rỗng nghĩa là version đó chỉ có PDF hoặc định dạng khác.

In [ ]:
# §10 — Quét thư mục version
ARCHIVE_NAMES = set(gfs.ARCHIVE_NAME.values())


def scan_paper_corpus(arxiv_id: str):
    out = {"arxiv_id": arxiv_id, "n_version_dirs": 0, "n_empty_versions": 0,
           "n_tex": 0, "n_bib": 0, "tex_bib_bytes": 0, "n_other": 0}
    bad = []
    pdir = utils.get_paper_dir(arxiv_id)
    for vdir in pdir.glob("v[0-9]*") if pdir.is_dir() else []:
        out["n_version_dirs"] += 1
        files = [f for f in vdir.rglob("*") if f.is_file()]
        if not files:
            out["n_empty_versions"] += 1
        for f in files:
            name = f.name.lower()
            if name.endswith((".tex", ".bib")):
                out["n_tex" if name.endswith(".tex") else "n_bib"] += 1
                out["tex_bib_bytes"] += f.stat().st_size
            elif not (KEEP_ARCHIVE and f.parent == vdir and f.name in ARCHIVE_NAMES):
                out["n_other"] += 1
                bad.append(f.relative_to(DATA_DIR).as_posix())
    return out, bad


scan_ids = status_df.loc[status_df["source"] != "pending", "arxiv_id"].tolist()
with ThreadPoolExecutor(max_workers=16) as pool:
    scanned = list(tqdm(pool.map(scan_paper_corpus, scan_ids), total=len(scan_ids), leave=False))
corpus_df = pd.DataFrame([s for s, _ in scanned], columns=list(scan_paper_corpus("x")[0]))
violations = [p for _, bad in scanned for p in bad]

print(f"Bài đã quét: {len(corpus_df)} | thư mục version: {corpus_df['n_version_dirs'].sum()} "
      f"(rỗng: {corpus_df['n_empty_versions'].sum()}) | .tex: {corpus_df['n_tex'].sum()} | .bib: {corpus_df['n_bib'].sum()}")
print(f"File không phải .tex/.bib: {len(violations)} (mong đợi 0)")
for p in violations[:20]:
    print("  ", p)

## §11. Thống kê dữ liệu

| Chỉ số | Định nghĩa |
|---|---|
| Bài cào thành công | có metadata **và** source `done` (mọi version đã thử tải và đã extract) |
| Tỷ lệ thành công | trên số ID trong range, và trên số bài arXiv thực sự có (metadata khác `not_found`) |
| Kích thước trước / sau khi bỏ hình | tổng byte mọi file trong archive (`sizes.full`) / tổng byte `.tex`+`.bib` giữ lại (`sizes.tex_bib`); tính trên các bài có source TeX |
| Số reference trung bình | trên các bài S2 trả reference (`ok`); kèm con số tính trên mọi bài đã hỏi S2 (`no_refs`/`not_found` tính 0) |
| Tỷ lệ lấy reference thành công | `ok` / số bài đã hỏi S2 |

In [ ]:
# §11 — Bảng per_paper (một dòng mỗi ID) + thống kê dữ liệu
def jsonable(obj):
    """Đổi kiểu numpy/pandas và NaN sang kiểu JSON chuẩn."""
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonable(v) for v in obj]
    if obj is pd.NA or obj is pd.NaT:
        return None
    if isinstance(obj, np.generic):
        obj = obj.item()
    if isinstance(obj, float) and math.isnan(obj):
        return None
    return obj


def mean_or_none(s: pd.Series):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(s.mean()) if len(s) else None


def ratio(a, b):
    return float(a) / float(b) if b else None


def source_info(entry) -> dict:
    entry = entry or {}
    st = Counter(v.get("status") for v in entry.get("versions", {}).values())
    sizes, stats = entry.get("sizes", {}), entry.get("stats", {})

    def total(stage, key):
        return sizes.get(stage, {}).get("totals", {}).get(key)

    via_s3 = sum(1 for v in entry.get("versions", {}).values() if str(v.get("via", "")).startswith("s3"))
    return {"n_versions_done": st.get("done", 0), "n_versions_no_tex": st.get("no_tex", 0),
            "n_versions_via_s3": via_s3,
            "n_versions_not_found": st.get("not_found", 0),
            "archive_bytes": total("archive_only", "archive_bytes"),
            "size_full_bytes": total("full", "extracted_bytes"),
            "size_tex_bib_bytes": total("tex_bib", "extracted_bytes"),
            "source_sleep_sec": stats.get("sleep_sec")}


timings = pd.DataFrame(read_jsonl(TIMINGS_FILE))
if timings.empty:
    timings = pd.DataFrame(columns=["run_id", "stage", "arxiv_id", "start", "end", "duration", "status"])
stage_time = (timings.groupby(["arxiv_id", "stage"])["duration"].sum().unstack()
              .reindex(columns=["metadata", "source", "refs"])
              .rename(columns=lambda s: f"time_{s}_sec"))

src_state = gfs.load_state().get("papers", {})
per_paper = status_df.copy()
per_paper = per_paper.join(pd.DataFrame([source_info(src_state.get(a)) for a in per_paper["arxiv_id"]],
                                        index=per_paper.index))
n_refs_arxiv = references_all.groupby("arxiv_id")["ref_arxiv_id"].count().rename("n_refs_arxiv")
per_paper = (per_paper.merge(n_refs_arxiv, left_on="arxiv_id", right_index=True, how="left")
             .merge(stage_time, left_on="arxiv_id", right_index=True, how="left"))
per_paper["time_total_sec"] = per_paper[[c for c in per_paper if c.startswith("time_")]].sum(axis=1, min_count=1)
per_paper["num_versions"] = pd.to_numeric(per_paper["num_versions"], errors="coerce")
per_paper["n_refs"] = pd.to_numeric(per_paper["n_refs"], errors="coerce")
per_paper["paper_success"] = (per_paper["metadata"] == "done") & (per_paper["source"] == "done")


def data_stats(pp: pd.DataFrame) -> dict:
    found = pp["metadata"] == "done"
    has_tex = pp["size_full_bytes"].notna() & (pd.to_numeric(pp["size_full_bytes"], errors="coerce") > 0)
    asked = pp["refs"].isin(["ok", "no_refs", "not_found"])
    before = pd.to_numeric(pp.loc[has_tex, "size_full_bytes"]).sum()
    after = pd.to_numeric(pp.loc[has_tex, "size_tex_bib_bytes"], errors="coerce").sum()
    n_refs_sum = pp.loc[pp["refs"] == "ok", "n_refs"].sum()
    return {
        "n_ids": len(pp),
        "n_papers_found": int(found.sum()),
        "n_metadata_not_found": int((pp["metadata"] == "not_found").sum()),
        "n_papers_success": int(pp["paper_success"].sum()),
        "success_rate_over_ids": ratio(pp["paper_success"].sum(), len(pp)),
        "success_rate_over_found": ratio(pp["paper_success"].sum(), found.sum()),
        "n_versions_total": int(pp.loc[found, "num_versions"].sum()),
        "n_versions_done": int(pp["n_versions_done"].sum()),
        "n_versions_no_tex": int(pp["n_versions_no_tex"].sum()),
        "n_versions_not_found": int(pp["n_versions_not_found"].sum()),
        "n_versions_via_s3": int(pp["n_versions_via_s3"].sum()),
        "n_papers_with_tex": int(has_tex.sum()),
        "avg_archive_bytes": mean_or_none(pp.loc[has_tex, "archive_bytes"]),
        "avg_size_before_bytes": mean_or_none(pp.loc[has_tex, "size_full_bytes"]),
        "avg_size_after_bytes": mean_or_none(pp.loc[has_tex, "size_tex_bib_bytes"]),
        "size_reduction_ratio": 1 - after / before if before else None,
        "n_refs_asked": int(asked.sum()),
        "refs_status_counts": pp["refs"].value_counts().to_dict(),
        "refs_success_rate": ratio((pp["refs"] == "ok").sum(), asked.sum()),
        "avg_refs_per_paper_ok": mean_or_none(pp.loc[pp["refs"] == "ok", "n_refs"]),
        "avg_refs_per_paper_asked": mean_or_none(pp.loc[asked, "n_refs"].fillna(0)),
        "n_references_all": int(n_refs_sum) if n_refs_sum == n_refs_sum else 0,
        "avg_refs_with_arxiv_per_paper_ok": mean_or_none(pp.loc[pp["refs"] == "ok", "n_refs_arxiv"].fillna(0)),
        "ref_arxiv_coverage": ratio(pp["n_refs_arxiv"].sum(), n_refs_sum),
    }


DATA_STATS = data_stats(per_paper)
range_stats = pd.DataFrame({label: data_stats(g) for label, g in per_paper.groupby("range_label")}).T
range_stats = range_stats.drop(columns=["refs_status_counts"])
versions_dist = per_paper.loc[per_paper["metadata"] == "done", "num_versions"].value_counts().sort_index()
primary_cat = papers_full["categories"].map(lambda c: c[0] if isinstance(c, list) and c else None)
category_counts = primary_cat.value_counts().rename_axis("primary_category").rename("n_papers")

pd.set_option("display.float_format", lambda x: f"{x:,.4f}".rstrip("0").rstrip("."))
display(pd.Series(jsonable(DATA_STATS), name="giá trị").to_frame())
display(range_stats)
print("Phân bố số version:", versions_dist.to_dict())
display(category_counts.head(15).to_frame())

## §12. Thống kê hiệu năng

| Chỉ số | Cách tính |
|---|---|
| Thời gian xử lý một bài | tổng thời lượng các stage của bài trong `timings.jsonl` (source; metadata và refs chia đều theo batch), cộng dồn qua các phiên |
| Thời gian trung bình mỗi bài | trung bình / median / p95 của cột trên trên các bài thành công; kèm wall time / số ID |
| Tổng thời gian entry discovery | tổng thời lượng mọi stage `metadata` |
| Wall time end-to-end | tổng (end − start) các phiên trong `runs.jsonl`; phiên bị ngắt đột ngột lấy mẫu monitor cuối làm end |
| RAM tối đa / trung bình | max / mean `rss_bytes` qua mọi `monitor_*.csv`; kèm `ru_maxrss` của kernel |
| Disk tối đa | max `data_dir_bytes` qua mọi `disk_*.csv` |
| Dung lượng output cuối | dung lượng `data/` trừ `run_logs/`, `.state/`, `tables/`, `report_exports/` |

In [ ]:
# §12 — Hiệu năng (cộng dồn mọi phiên)
def load_csv_logs(prefix: str) -> pd.DataFrame:
    frames = []
    for p in sorted(LOG_DIR.glob(f"{prefix}_*.csv")):
        try:
            df = pd.read_csv(p, on_bad_lines="skip")
        except pd.errors.EmptyDataError:
            continue
        df["run_id"] = p.stem.split("_", 1)[1]
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["t", "run_id"])


def load_runs(monitor: pd.DataFrame) -> pd.DataFrame:
    recs = read_jsonl(RUNS_FILE)
    ends = {r["run_id"]: r for r in recs if r.get("event") == "end"}
    rows = []
    for s in (r for r in recs if r.get("event") == "start"):
        rid, e = s["run_id"], ends.get(s["run_id"])
        if e:
            end_t, reason = e["t"], e.get("stop_reason")
        else:  # runtime bị ngắt trước khi ghi dòng end
            last = monitor.loc[monitor["run_id"] == rid, "t"].max() if len(monitor) else None
            end_t = last if last == last and last is not None else s["t"]
            reason = "killed (end = mẫu monitor cuối)"
        m = s.get("machine", {})
        rows.append({"run_id": rid, "start": s["t"], "end": end_t, "wall_sec": end_t - s["t"],
                     "stop_reason": reason, "todo_arxiv": s.get("todo", {}).get("arxiv"),
                     "todo_refs": s.get("todo", {}).get("refs"), "colab": m.get("colab"),
                     "cpu_model": m.get("cpu_model"), "cpu_count": m.get("cpu_count"),
                     "ram_total_bytes": m.get("ram_total_bytes"),
                     "peak_rss_kernel_bytes": (e or {}).get("peak_rss_kernel_bytes")})
    return pd.DataFrame(rows)


monitor_df = load_csv_logs("monitor")
disk_df = load_csv_logs("disk")
runs_df = load_runs(monitor_df)

ok_time = per_paper.loc[per_paper["paper_success"], "time_total_sec"].dropna()
wall = float(runs_df["wall_sec"].sum()) if len(runs_df) else 0.0
output_bytes = dir_size(DATA_DIR, exclude=NON_CORPUS_DIRS)

PERF_STATS = {
    "n_sessions": len(runs_df),
    "wall_time_sec": wall,
    "time_per_paper_mean_sec": float(ok_time.mean()) if len(ok_time) else None,
    "time_per_paper_median_sec": float(ok_time.median()) if len(ok_time) else None,
    "time_per_paper_p95_sec": float(ok_time.quantile(0.95)) if len(ok_time) else None,
    "wall_time_per_id_sec": wall / len(per_paper) if len(per_paper) else None,
    "entry_discovery_total_sec": float(timings.loc[timings["stage"] == "metadata", "duration"].sum()),
    "source_total_sec": float(timings.loc[timings["stage"] == "source", "duration"].sum()),
    "refs_total_sec": float(timings.loc[timings["stage"] == "refs", "duration"].sum()),
    "throughput_papers_per_hour": per_paper["paper_success"].sum() / (wall / 3600) if wall else None,
    "rss_max_bytes": float(monitor_df["rss_bytes"].max()) if "rss_bytes" in monitor_df else None,
    "rss_mean_bytes": float(monitor_df["rss_bytes"].mean()) if "rss_bytes" in monitor_df else None,
    "peak_rss_kernel_bytes": float(runs_df["peak_rss_kernel_bytes"].max()) if len(runs_df) else None,
    "disk_max_bytes": float(disk_df["data_dir_bytes"].max()) if "data_dir_bytes" in disk_df else None,
    "output_size_bytes": output_bytes,
    "tables_size_bytes": dir_size(TABLE_DIR),
}
S3_CHUNKS = s3s.load_chunk_state()
PERF_STATS["s3_chunks_read"] = len(S3_CHUNKS)
PERF_STATS["s3_bytes_read"] = s3s.bytes_read_total(S3_CHUNKS)
PERF_STATS["s3_read_sec"] = sum(c.get("seconds", 0.0) for c in S3_CHUNKS.values())
PERF_STATS["s3_cost_usd_estimate"] = PERF_STATS["s3_bytes_read"] / 1e9 * s3s.EGRESS_USD_PER_GB

display(pd.Series(jsonable(PERF_STATS), name="giá trị").to_frame())
print(f"Wall time: {fmt_secs(wall)} qua {len(runs_df)} phiên | RSS max {fmt_bytes(PERF_STATS['rss_max_bytes'])} | "
      f"disk max {fmt_bytes(PERF_STATS['disk_max_bytes'])} | output {fmt_bytes(output_bytes)}")
display(runs_df)

In [ ]:
# §12 — Vài hình kiểm tra nhanh (hình chính thức do Jupyter Book vẽ từ report_exports/)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
if len(monitor_df):
    m = monitor_df.sort_values("t")
    axes[0].plot((m["t"] - m["t"].min()) / 3600, m["rss_bytes"] / 2**20, lw=0.8)
axes[0].set(title="RSS theo thời gian", xlabel="giờ (từ phiên đầu)", ylabel="MB")
if len(ok_time):
    axes[1].hist(ok_time, bins=40)
axes[1].set(title="Thời gian xử lý mỗi bài thành công", xlabel="giây", ylabel="số bài")
sizes = [DATA_STATS["avg_size_before_bytes"] or 0, DATA_STATS["avg_size_after_bytes"] or 0]
axes[2].bar(["trước khi bỏ hình", "chỉ .tex/.bib"], [s / 2**10 for s in sizes])
axes[2].set(title="Kích thước trung bình mỗi bài", ylabel="KB")
plt.tight_layout()
plt.show()

## §13. Xuất số liệu cho Jupyter Book

Các file nhỏ trong `data/report_exports/` được chép sang repo Jupyter Book. Book đọc các file này và **vẽ hình lúc build**, vì đề không chấp nhận ảnh chụp dán vào.

In [ ]:
# §13 — Ghi report_exports/
EXPORT_README = """# report_exports

Số liệu do `src/scraper.ipynb` (Phần IV) sinh ra, dùng cho Jupyter Book.

| File | Nội dung |
|---|---|
| `summary_stats.json` | `data` (thống kê dữ liệu, §11), `performance` (hiệu năng, §12), `versions_distribution`, `config`, `generated_at` |
| `per_paper.csv` | một dòng mỗi arXiv ID: `range_label`; trạng thái `metadata` / `source` / `refs`; `num_versions`, `n_versions_done/no_tex/not_found`, `n_versions_via_s3`; `archive_bytes` (nén), `size_full_bytes` (trước khi bỏ hình), `size_tex_bib_bytes` (sau); `n_refs`, `n_refs_arxiv`; `time_metadata_sec`, `time_source_sec`, `time_refs_sec` (chia đều theo batch), `time_total_sec`; `source_sleep_sec`; `paper_success` |
| `range_stats.csv` | các chỉ số của `summary_stats.data` tính riêng cho từng range |
| `category_counts.csv` | số bài theo category chính (category đầu tiên) |
| `runs.csv` | mỗi phiên chạy: `start`, `end` (epoch giây), `wall_sec`, `stop_reason`, việc của phiên, thông tin máy, `peak_rss_kernel_bytes` |
| `monitor.csv` | RSS gộp theo phút: `run_id`, `minute` (epoch giây), `rss_max_bytes`, `rss_mean_bytes` |
| `disk.csv` | dung lượng `data/` theo thời gian: `run_id`, `t`, `data_dir_bytes`, `scan_sec` |
| `s3_chunks.csv` | mỗi chunk S3 đã đọc: `chunk`, `status`, `bytes_read`, `seconds`, `stored`, `n_missing` |
"""

summary = {
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "config": run_config(),
    "data": DATA_STATS,
    "performance": PERF_STATS,
    "versions_distribution": versions_dist.to_dict(),
}
(EXPORT_DIR / "summary_stats.json").write_text(json.dumps(jsonable(summary), indent=2, ensure_ascii=False), encoding="utf-8")

export_cols = ["arxiv_id", "range_label", "metadata", "source", "refs", "num_versions", "n_versions_done",
               "n_versions_no_tex", "n_versions_not_found", "n_versions_via_s3", "archive_bytes", "size_full_bytes", "size_tex_bib_bytes",
               "n_refs", "n_refs_arxiv", "time_metadata_sec", "time_source_sec", "time_refs_sec", "time_total_sec",
               "source_sleep_sec", "paper_success"]
per_paper[export_cols].to_csv(EXPORT_DIR / "per_paper.csv", index=False)
range_stats.to_csv(EXPORT_DIR / "range_stats.csv", index_label="range_label")
category_counts.to_csv(EXPORT_DIR / "category_counts.csv")
runs_df.to_csv(EXPORT_DIR / "runs.csv", index=False)
if len(monitor_df):
    (monitor_df.assign(minute=(monitor_df["t"] // 60 * 60).astype("int64"))
     .groupby(["run_id", "minute"])["rss_bytes"].agg(rss_max_bytes="max", rss_mean_bytes="mean")
     .reset_index().to_csv(EXPORT_DIR / "monitor.csv", index=False))
disk_df.to_csv(EXPORT_DIR / "disk.csv", index=False)
(pd.DataFrame([{"chunk": k, "status": v.get("status"), "bytes_read": v.get("bytes_read", 0),
                "seconds": v.get("seconds", 0.0), "stored": v.get("stored", 0), "n_missing": len(v.get("missing", []))}
               for k, v in S3_CHUNKS.items()], columns=["chunk", "status", "bytes_read", "seconds", "stored", "n_missing"])
 .to_csv(EXPORT_DIR / "s3_chunks.csv", index=False))
(EXPORT_DIR / "README.md").write_text(EXPORT_README, encoding="utf-8")

export_size = dir_size(EXPORT_DIR)
for p in sorted(EXPORT_DIR.iterdir()):
    print(f"{p.name:22s} {fmt_bytes(p.stat().st_size):>10}")
print(f"Tổng: {fmt_bytes(export_size)}" + ("  (CẢNH BÁO: > 10 MB)" if export_size > 10 * 2**20 else ""))